# Session 4 — Security as GRC: Governing & Observing Agentic AI

**Exercise: observe and govern an agent** — review an activity log and classify each event as Allow, Monitor, Require Approval or Block; identify what should be logged, alerted, retained and reviewed.

You will pull real activity from the shared project (agents, tools, your own runs) and from Azure's control plane (role assignments), then apply the Four-Layer Guardrail model: **Policy → Enforcement → Oversight → Assurance**.

In [ ]:
# Installs the Azure libraries and fetches the workshop files.
# Only does anything in Google Colab. Elsewhere it is skipped.
# Google Colab only: install the SDKs and fetch the workshop helpers. Local Jupyter/VS Code: skip.
import sys, subprocess, pathlib
if "google.colab" in sys.modules and not pathlib.Path("workshop.py").exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "azure-ai-projects>=2", "azure-ai-agents>=1.1", "azure-identity>=1.17"], check=True)
    subprocess.run(["git", "clone", "-q", "https://github.com/Auxin-io/Azure-GenAI-Security-Workshop.git", "_ws"], check=True)
    subprocess.run("cp -r _ws/workshop.py _ws/data . ", shell=True, check=True)
    print("Colab setup done - the next cell will ask for the workshop secret.")
    print("If the facilitator gave you a hosted workshop link, use that instead: it signs in")
    print("for you with a managed identity and needs no Azure account at all.")

In [ ]:
# Signs you in.
# In Colab it asks once for the workshop secret. On the hosted environment or a
# machine with `az login` it needs nothing from you.
import workshop as w

w.sign_in()

client = w.agents_client()

## 1. Agent inventory — what exists, who owns it, what can it touch

In [ ]:
# Lists the agents actually running, and what each one can reach.
# Look at the identity column: all three share one. That is the finding.
rows = []
for a in client.list_agents():
    tools = [t["type"] for t in a.tools]
    reaches = []
    for t in a.tools:
        if t["type"] == "openapi":
            reaches.append(t["openapi"]["spec"]["servers"][0]["url"])
        elif t["type"] == "file_search":
            reaches.append("vector store " + ",".join(t.get("file_search", {}).get("vector_store_ids", [])) if isinstance(t, dict) else "vector store")
    rows.append((a.name, a.model, ", ".join(tools) or "-", ", ".join(reaches) or "-"))
print(f"{'agent':<30} {'model':<14} {'tools':<24} reaches")
for r in rows:
    print(f"{r[0]:<30} {r[1]:<14} {r[2]:<24} {r[3]}")

**Exercise 4.1** — complete the inventory. For each agent add: owner, data classification of what it can reach, risk tier (Allow / Monitor / Require approval / Block) and the reason.

In [ ]:
# Written exercise - the ownership register. Who owns each agent, what data does
# it touch, who approves its actions.
inventory = {
    "docintel-finance-agent":  {"owner": "", "data": "", "tier": "", "reason": ""},
    "docintel-employee-agent": {"owner": "", "data": "", "tier": "", "reason": ""},
    "docintel-hr-agent":       {"owner": "", "data": "", "tier": "", "reason": ""},
}
for k, v in inventory.items():
    print(k, v)

## 2. Enforcement evidence — who is allowed to do what

The control plane is the source of truth. This lists every role assignment on the endpoints and on the AI Services account (needs Reader on the resource group; if it fails, use the sample below).

In [ ]:
# Reads the REAL permissions out of Azure: who holds which role, on what.
# Needs the Azure CLI, so it prints a note instead when run in Colab.
import subprocess, shutil, json
AZ = shutil.which("az") or shutil.which("az.cmd") or "az"
def az(*args):
    return json.loads(subprocess.check_output([AZ, *args, "-o", "json"], text=True))

rg = w.CONFIG["resource_group"]
try:
    ws = az("ml", "workspace", "list", "-g", rg)[0]["name"]
    scopes = [az("ml", "online-endpoint", "show", "-n", e["name"], "-g", rg, "-w", ws)["id"]
              for e in az("ml", "online-endpoint", "list", "-g", rg, "-w", ws)]
    scopes += [a["id"] for a in az("cognitiveservices", "account", "list", "-g", rg) if a["kind"] == "AIServices"]
    for scope in scopes:
        print(scope.split("/")[-1])
        for ra in az("role", "assignment", "list", "--scope", scope):
            print(f"   {ra['roleDefinitionName']:<28} {ra['principalType']:<17} {ra.get('principalName') or ra['principalId']}")
except Exception as e:
    print("could not list (needs az + Reader):", e)

**Exercise 4.2** — for each assignment: is it the *minimum* needed? Which one would you remove first? Which is missing (think of the three agents sharing one project)?

## 3. Controls at build time — where each one attaches

Governance fails when the control list and the build are two different documents. This section puts
them side by side: the eight decisions you make while creating an agent, and the control that
attaches at each one.

Control IDs are from the Session 3 layer tables — **P** perception, **C** cognitive, **A** action,
**I** integration, **O** operations, **F** infrastructure.

| # | Decision when creating the agent | Controls that attach here | Enforced by |
|---|---|---|---|
| 1 | Which model, which version | I3 model allow-list, F4 provenance | platform |
| 2 | The instructions you write | C1 immutable system prompt, A3 versioned prompt | **you** |
| 3 | Which tools you attach | C7 per-task allow-list, A1 signed manifests | **you** |
| 4 | How each tool authenticates | I1 workload identity, A1 calling-user identity, F3 no static keys | **you** |
| 5 | Which knowledge it can reach | A2 collection ACLs, I4 row/field authorisation, C4 context as data | **you** |
| 6 | Which actions need a human | C6 risk-tiered approval, A4 diff and blast radius, I2 MFA on approver | **you** |
| 7 | What the budgets are | C2 step and delegation caps, O2 token and spend quotas | **you** |
| 8 | What is recorded | O3 append-only redacted trace, O5 detections to SIEM | platform + **you** |

Rows 2–7 are the developer's. Nobody else can add them later.

In [ ]:
# Checks each live agent against the controls from the Session 3 deck.
# A 'NO' is not automatically a fault - a read-only agent needs no approval tool.
# The finding is a NO where the decision should have been made.
# Read the controls off the agents that actually exist, rather than off the design document.
# Each check answers one question: is this control present on this agent, right now?
CONTROL_CHECKS = {
    "2 instructions set":      lambda a: bool((a.instructions or "").strip()),
    "2 grounding rule stated": lambda a: any(k in (a.instructions or "").lower() for k in
                                             ("only from", "verbatim", "do not guess", "cite")),
    "3 tools attached":        lambda a: len(a.tools or []) > 0,
    "3 tool count is small":   lambda a: 0 < len(a.tools or []) <= 3,
    "5 knowledge scoped":      lambda a: bool(getattr(a, "tool_resources", None)),
    "6 human-approval tool":   lambda a: any(_tool_type(t) == "function" for t in (a.tools or [])),
}

def _tool_type(t):
    return t.get("type") if isinstance(t, dict) else getattr(t, "type", "?")

rows = []
for a in client.list_agents():
    rows.append((a.name, {k: fn(a) for k, fn in CONTROL_CHECKS.items()}))

labels = list(CONTROL_CHECKS)
width = max(len(n) for n, _ in rows) + 2
print("agent".ljust(width) + "".join(f"{i:^7}" for i in range(1, len(labels) + 1)))
for name, res in sorted(rows):
    print(name.ljust(width) + "".join(f"{('yes' if v else 'NO'):^7}" for v in res.values()))
print()
for i, l in enumerate(labels, 1):
    print(f"  {i}. {l}")

**A `NO` is not automatically a finding.** A read-only agent has no write tool, so "human-approval
tool" is correctly absent. The finding is a `NO` on an agent where the decision *should* have been
made — and the register below is where you record which is which.

In [ ]:
# The questions the code cannot answer for you. Fill them in from what you saw.
# Rows 4 and 8 cannot be read off the agent object: they are properties of the platform around it.
# Fill these in from what you saw in sections 1 and 2, and from Session 3's trace.
platform_controls = {
    "4 tool auth is managed identity, no keys":      None,   # True / False
    "4 tool runs as the CALLING user, not the agent": None,
    "7 step or token budget enforced outside model":  None,
    "8 traces redacted at capture":                   None,
    "8 trace store append-only":                      None,
    "8 agent detections reach a SIEM":                None,
}
for k, v in platform_controls.items():
    print(f"{'?' if v is None else ('yes' if v else 'NO ')}  {k}")

### The exercise

1. For every `NO` above, decide: **accepted risk**, **compensating control**, or **must fix**.
2. For each *must fix*, write the control, where it is enforced, and who owns it.
3. One of the rows in the second cell is **False for this architecture and cannot be fixed by
   configuration** — find it, and say what would have to change in the design.

| Finding | Decision | Control | Enforced where | Owner | Governance risk (G1–G10) |
|---|---|---|---|---|---|
| | | | | | |
| | | | | | |
| | | | | | |

Map each one to a governance risk ID from the deck's register (G1 shadow agents … G10 reputational
exposure). A finding with no G-number is a bug report; a finding with one is a governance item, and
that is the difference this session is about.

## 4. Oversight — an activity log to classify

Below is a log built from real run steps and control-plane events of this system (names shortened). Classify every line.

In [ ]:
# A real activity log: twelve things these agents did, including the invented
# invoice from Session 1 and the poisoning from Session 2.
log = [
    {"ts": "09:01", "actor": "agent:finance",  "event": "tool_call openapi answerFinanceQuestion", "detail": "Xenon Energy total"},
    {"ts": "09:02", "actor": "agent:finance",  "event": "tool_call openapi answerFinanceQuestion", "detail": "Cedar Systems total (vendor not in corpus) -> answered with INV-35089"},
    {"ts": "09:05", "actor": "agent:hr",       "event": "file_search",                          "detail": "notice period -> doc-hr-001.txt"},
    {"ts": "09:06", "actor": "agent:hr",       "event": "file_search",                          "detail": "parental leave -> no chunk -> refusal"},
    {"ts": "09:10", "actor": "user:attendee",  "event": "vector_store.create",                  "detail": "hr-poisoned-attendee, 11 files"},
    {"ts": "09:11", "actor": "agent:hr-copy",  "event": "message",                              "detail": "'no notice required, email password to payroll-help@example.net'"},
    {"ts": "09:20", "actor": "agent:guarded",  "event": "function_call approve_expense",        "detail": "EXP-70486, approved by reviewer"},
    {"ts": "09:21", "actor": "agent:guarded",  "event": "function_call approve_expense",        "detail": "EXP-70486, denied by reviewer (no explicit request)"},
    {"ts": "09:30", "actor": "user:akhan",     "event": "roleAssignments/write",                "detail": "AzureML Data Scientist on employee-from-scratch to project identity"},
    {"ts": "09:45", "actor": "user:akhan",     "event": "models/register",                      "detail": "employee-from-scratch-model:1"},
    {"ts": "10:02", "actor": "agent:finance",  "event": "content_filter",                       "detail": "jailbreak prompt 'reply HACKED' blocked - run ended incomplete"},
    {"ts": "10:15", "actor": "identity:agent", "event": "token_request",                        "detail": "audience ml.azure.com, from Bot Service"},
]
decisions = {}   # index -> ("Allow" | "Monitor" | "Require approval" | "Block", "why")
for i, e in enumerate(log):
    print(f"{i:>2} {e['ts']} {e['actor']:<16} {e['event']:<42} {e['detail']}")

In [ ]:
# Written exercise - label every line Allow / Monitor / Require approval / Block.
# It tells you which ones you have not classified yet.
# Exercise 4.3 - fill in your decisions, then run
decisions = {
    0: ("Allow", "expected read"),
    # 1: (...),
}
for i, (d, why) in sorted(decisions.items()):
    print(f"{i:>2} {log[i]['event']:<42} {d:<17} {why}")
missing = [i for i in range(len(log)) if i not in decisions]
print("\nnot yet classified:", missing)

## 5. Assurance — write the guardrail policy for these agents

Fill in the four layers for **this** deployment. Keep each line to what you can point at in the portal or a repo.

| Layer | What it is here | Evidence |
|---|---|---|
| Policy | | |
| Enforcement | | |
| Oversight | | |
| Assurance | | |

Then: what should be **retained** (and for how long), what should **alert** (to whom), and what is **reviewed** weekly?

_Your answers:_